# M5 경제표현 + 양성 구매금액 가중학습: H&M 2년 test-only seed 42

Dunnhumby M5와 같은 A~F 여섯 arm, 경제표현, 손실, rho, lambda와 100 epoch을 사용합니다. H&M의 기존 train과 validation을 합쳐 2020-09-08까지 학습하고, 2020-09-09~15 test를 마지막 checkpoint에서 한 번만 평가합니다. 2020-09-16~22는 사용하지 않습니다. 단일 seed 결과이므로 안정성·유의성·일반화를 주장하지 않으며 이 결과로 수식이나 하이퍼파라미터를 다시 선택하지 않습니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import importlib, os, shutil, subprocess, sys

REVIEWED_SHA = 'd0df4819560c35d30189e913a223fe432c30108c'
REPO_URL = 'https://github.com/jung-un/clv-m2-lightgcn-runner.git'
os.chdir('/content')
repo = Path('/content/clv-m2-lightgcn-runner')
clone_errors = []
for clone_attempt in range(1, 4):
    if repo.exists():
        shutil.rmtree(repo)
    result = subprocess.run(
        ['git', 'clone', REPO_URL, str(repo)],
        text=True, capture_output=True,
    )
    if result.returncode == 0:
        break
    clone_errors.append(result.stderr.strip())
    print(f'GitHub clone {clone_attempt}/3 실패:', result.stderr.strip())
else:
    raise RuntimeError('GitHub clone 3회 실패:\n' + '\n'.join(clone_errors))
subprocess.run(['git', '-C', str(repo), 'checkout', '-q', REVIEWED_SHA], check=True)
actual_sha = subprocess.check_output(
    ['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True
).strip()
assert actual_sha == REVIEWED_SHA, (actual_sha, REVIEWED_SHA)
for module_name in tuple(sys.modules):
    if module_name.startswith(('lightgcn_', 'clv_')):
        del sys.modules[module_name]
importlib.invalidate_caches()
%cd /content/clv-m2-lightgcn-runner
print('실행 코드 고정 완료:', actual_sha)

In [ ]:
import json
import torch
from lightgcn_clv_m5_economic_positive_weight_test import (
    configure_m5_economic_positive_test_run,
    preflight_summary,
    run_m5_economic_positive_test,
)

assert torch.cuda.is_available(), '런타임 유형에서 GPU를 선택하세요.'
cfg = configure_m5_economic_positive_test_run(
    dataset='hm',
    seeds=(42,),
    batch_size=131_072,
    out_dir='/content/drive/MyDrive/논문/data/results_v3_hm_m5_economic_positive_weighting_test_seed42_v1',
)
summary = preflight_summary(cfg)
assert cfg.seeds == (42,)
assert summary['period'] == 'full_history_about_2_years'
assert summary['training_data'] == 'through 2020-09-08 (former train + validation)'
assert summary['test_data'] == '2020-09-09--15'
assert summary['validation_constructed'] is False
assert summary['holdout_evaluation'] is False
print(json.dumps(summary, ensure_ascii=False, indent=2))

In [ ]:
result_df = run_m5_economic_positive_test(cfg)

In [ ]:
from IPython.display import display

def show(frame):
    view = frame.copy()
    view.attrs = {}
    display(view)

print('1) H&M 2년 seed 42 test 절대지표')
show(result_df)
print('2) 모델별 seed 42 지표표')
show(result_df.attrs['mean'])
print('3) 동일 seed 대조군 비교')
show(result_df.attrs['comparison'])
print('4) 동일 seed 대응차')
show(result_df.attrs['paired_mean'])
print("5) M2 × M4' 상호작용")
show(result_df.attrs['interaction'])
print('6) 사전 고정 기준의 기술적 판독')
print(json.dumps(result_df.attrs['descriptive_reading'], ensure_ascii=False, indent=2))
print('7) 저장 파일')
print(json.dumps(result_df.attrs['result_paths'], ensure_ascii=False, indent=2))